In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from scripts.VectorFieldEmbedder import *
from scripts.plotting import *

X = np.load("data/pancreas/expression.npy")
X_umap = np.load("data/pancreas/expression_umap.npy")
V = np.load("data/pancreas/velocity.npy")
cell_type = np.load("data/pancreas/clusters.npy", allow_pickle=True)

def scale_columns(X):
    return X / np.std(X, axis=0, keepdims=True)

X = scale_columns(X)
V = scale_columns(V)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scripts.VectorFieldEmbedder import VectorFieldEmbedder
from scripts.plotting import plot_velocity_streamplot

# Define parameter grids
knn_k_values = [20, 30, 40, 50]
min_dist_values = [0.1, 0.3, 0.5, 0.7]

fig, axes = plt.subplots(4, 4, figsize=(22, 22))

for i, knn_k in enumerate(knn_k_values):
    for j, min_dist in enumerate(min_dist_values):
        # Update UMAP params
        umap_params = {"min_dist": min_dist}

        # Initialize embedder
        emb = VectorFieldEmbedder(
            X, V,
            dist_method="phase",
            dof=60,
            method="umap",
            embed_kwargs=umap_params,
            alpha=0.5,
            max_tps_points=4000,
            knn_k=knn_k
        )
        emb.initialize_embedding(seed=123)

        # Plot each combination into subplot
        ax = axes[i, j]
        plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=cell_type,
            grid_density=1,
            stream_density=1.2,
            scatter_size=50,
            scatter_alpha=0.18,
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=40,
            ax=ax
        )
        ax.set_title(f"knn_k={knn_k}, min_dist={min_dist}")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scripts.VectorFieldEmbedder import VectorFieldEmbedder
from scripts.plotting import plot_velocity_streamplot

# Fixed parameters
knn_k = 30
min_dist = 0.5
alpha_values = [0.0, 0.3, 0.5, 1.0]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for i, alpha in enumerate(alpha_values):
    umap_params = {"min_dist": min_dist}

    emb = VectorFieldEmbedder(
        X, V,
        dist_method="phase",
        dof=30,
        method="umap",
        embed_kwargs=umap_params,
        alpha=alpha,
        max_tps_points=4000,
        knn_k=knn_k
    )
    emb.initialize_embedding(seed=123)

    ax = axes[i]
    plot_velocity_streamplot(
        X_2d=emb.X_emb,
        tps_vf=emb.tps_vf,
        scatter_color=cell_type,
        grid_density=1,
        stream_density=1.2,
        scatter_size=50,
        scatter_alpha=0.18,
        aspect=1,
        vmin=0.0,
        vmax=1.0,
        cmap="tab10",
        grid_size=40,
        ax=ax
    )
    ax.set_title(f"alpha={alpha}")

plt.tight_layout()
plt.show()
